In [1]:
import pandas
import sklearn.compose
import sklearn.preprocessing
import sklearn.model_selection

data = pandas.read_csv("churn.csv")
data = data.drop(columns = ['customerID'])
data = data[data["TotalCharges"] != ' ']
data["TotalCharges"] = data["TotalCharges"].astype(float)

cols_to_replace = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
data[cols_to_replace] = data[cols_to_replace].replace(['No phone service', 'No internet service'], 'No')

X = data.drop(columns = ["Churn"])
y = (data["Churn"] == "Yes").to_numpy()
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size = 0.1, stratify = y, random_state = 42
)

numeric = [ "tenure", "MonthlyCharges", "TotalCharges" ]
categorical = list(set(X_train.columns) - set(numeric))
ct = sklearn.compose.ColumnTransformer(
    transformers = [
        ("numeric", 'passthrough', numeric),
        ("categorical", sklearn.preprocessing.OneHotEncoder(dtype = 'bool'), categorical)
    ]
)
X_train = pandas.DataFrame(ct.fit_transform(X_train), columns = ct.get_feature_names_out())
X_test = pandas.DataFrame(ct.transform(X_test), columns = ct.get_feature_names_out())

categorical = [ feature for feature in ct.get_feature_names_out() if feature.startswith("categorical__") ]
X_train[categorical] = X_train[categorical].astype(bool)
X_test[categorical] = X_test[categorical].astype(bool)

y_train = pandas.Series(y_train)
y_test = pandas.Series(y_test)

In [2]:
from lazyfca import LazyFCA

classifier = LazyFCA(
    min_pos_for_pos_clas = 5,
    pos_coef_for_pos_clas = 2.75,
    min_neg_for_neg_clas = 10,
    neg_coef_for_neg_clas = 0.25,
    pos_clas_coef = 1.0
)
classifier.fit(X_train, y_train)

In [3]:
y_pred = classifier.predict(X_test)

from utils import estimate_quality
estimate_quality(y_pred, y_test)

100%|██████████| 704/704 [00:44<00:00, 15.97it/s]


{'Accuracy': 0.7556818181818182,
 'Precision': 0.5265017667844523,
 'Recall': 0.7967914438502673,
 'AUC-ROC': 0.8240724459293125,
 'F1-score': 0.6340425531914894,
 'True Positive': 149,
 'True Negative': 383,
 'False Positive': 134,
 'False Negative': 38,
 'True Negative Rate (Specificity)': 0.7408123791102514,
 'Negative Predictive Value': 0.9097387173396675,
 'False Positive Rate': 0.25918762088974856,
 'False Discovery Rate': 0.4734982332155477}

In [12]:
classifier.explain(X_test.iloc[502]).display()

,Type,Supporters,Opposers,Supporters covered,Opposers covered
0,POSITIVE,1682,4646,5,6
1,POSITIVE,1682,4646,7,14
2,POSITIVE,1682,4646,341,874
3,POSITIVE,1682,4646,6,12
4,POSITIVE,1682,4646,5,13
...,...,...,...,...,...
3547,NEGATIVE,4646,1682,244,26
3548,NEGATIVE,4646,1682,25,0
3549,NEGATIVE,4646,1682,32,0
3550,NEGATIVE,4646,1682,74,1
